1. Import, model, dan fungsi bantu
P berisi koefisien keuntungan. A berisi koefisien kendala, sedangkan b berisi batas kanan kendala. A @ x menghasilkan pemakaian untuk kedua kendala sekaligus.

    **Fungsi feasible** memeriksa kelayakan terhadap kendala asli. Syarat bilangan bulat diperiksa terpisah pada masing-masing metode. Fungsi selesaikan_lp menerima batas variabel dari cabang yang sedang diperiksa.

    **menyelesaikan minimisasi.** Karena soal meminta maksimum $Z$, koefisien tujuan dimasukkan sebagai -P, yaitu [-7, -6]. Meminimalkan $-Z$ setara dengan memaksimalkan $Z$.

    **TOL = 1e-7** digunakan untuk perbandingan numerik, karena hasil solver berupa bilangan desimal yang bisa memiliki selisih sangat kecil dari nilai matematisnya.


In [1]:
import numpy as np  # NumPy dipakai untuk array dan operasi matriks.
from scipy.optimize import linprog  # Solver untuk LP yang variabelnya masih kontinu.
from math import floor, ceil  # floor membulatkan ke bawah; ceil ke atas.
from collections import deque  # deque menyimpan antrean cabang yang akan diperiksa.

P = np.array([7.0, 6.0])  # Koefisien keuntungan untuk x1 dan x2.
A = np.array([[2.0, 3.0], [6.0, 5.0]])  # Setiap baris berisi koefisien satu kendala.
b = np.array([12.0, 30.0])  # Batas kanan kedua kendala.
TOL = 1e-7  # Toleransi kecil untuk perbandingan bilangan desimal dari solver.

def keuntungan(x):  # Menghitung Z untuk pasangan nilai x1 dan x2.
    return float(P @ x)  # @ menghitung 7*x1 + 6*x2; float membuat hasil desimal.

def feasible(x):  # Memeriksa syarat tidak negatif dan kedua kendala asli.
    return bool(np.all(x >= -TOL) and np.all(A @ x <= b + TOL))  # Semua syarat harus benar.

def selesaikan_lp(batas):  # Menyelesaikan LP dengan batas variabel pada cabang tertentu.
    return linprog(c=-P, A_ub=A, b_ub=b, bounds=batas, method="highs")  # Min(-Z) = max(Z).

2. Solusi LP tanpa syarat integer
Untuk langkah awal, variabel boleh berbentuk pecahan. Batas (0, None) berarti nilai variabel paling kecil 0 dan tidak diberi batas atas terpisah. Kedua kendala produksi tetap berlaku.
Hasil yang diharapkan: $x_1=3{,}75$, $x_2=1{,}5$, dan $Z=35{,}25$. Solusi ini menjadi batas atas keuntungan untuk model integer, tetapi belum dapat digunakan sebagai jumlah mesin utuh.


In [2]:
hasil_lp = selesaikan_lp([(0, None), (0, None)])  # x1 dan x2 >= 0; None berarti tanpa batas atas.
if not hasil_lp.success:  # Pastikan solver berhasil sebelum membaca solusi.
    raise RuntimeError(hasil_lp.message)  # Hentikan program jika LP gagal diselesaikan.
x_lp = hasil_lp.x  # Ambil nilai optimal x1 dan x2 yang boleh berbentuk pecahan.
z_lp = keuntungan(x_lp)  # Hitung keuntungan pada solusi LP.
print("SOLUSI LP")  # Judul hasil LP.
print(f"x1 = {x_lp[0]:.4f}; x2 = {x_lp[1]:.4f}; Z = {z_lp:.4f}")  # Tampilkan empat angka desimal.

SOLUSI LP
x1 = 3.7500; x2 = 1.5000; Z = 35.2500


3. Round Off dengan pemeriksaan kelayakan
Pembulatan terdekat langsung ke $(4,2)$ tidak feasible: kendala pertama menjadi $14>12$, sedangkan kendala kedua menjadi $34>30$.
Untuk mengikuti contoh Round Off pada slide, kode memeriksa empat kombinasi pembulatan ke bawah/atas terdekat: $(3,1)$, $(3,2)$, $(4,1)$, dan $(4,2)$. Pilihan terbaik yang memenuhi kendala adalah $(4,1)$, dengan $Z=34$.

    Metode ini merupakan heuristik: hasilnya terbaik di antara kombinasi pembulatan yang diperiksa, dan belum tentu optimal untuk seluruh model integer. Jika tidak ada kandidat pembulatan yang feasible pada model lain, kode melaporkan kegagalan.


In [3]:
def round_off(x_awal):  # Periksa kombinasi pembulatan bawah/atas terdekat dari solusi LP.
    pilihan_x1 = sorted({floor(x_awal[0]), ceil(x_awal[0])})  # Kandidat x1: 3 dan 4.
    pilihan_x2 = sorted({floor(x_awal[1]), ceil(x_awal[1])})  # Kandidat x2: 1 dan 2.
    terbaik_x = None  # Belum ada pasangan pembulatan yang dipilih.
    terbaik_z = -np.inf  # Nilai awal lebih kecil daripada setiap keuntungan calon solusi.
    print("\nROUND OFF: KOMBINASI PEMBULATAN TERDEKAT")  # Judul metode pertama.
    print(" x1  x2       Z    Kendala 1    Kendala 2    Feasible")  # Judul kolom.
    for x1 in pilihan_x1:  # Periksa setiap kandidat x1.
        for x2 in pilihan_x2:  # Pasangkan x1 dengan setiap kandidat x2.
            x = np.array([x1, x2], dtype=int)  # Buat pasangan bilangan bulat.
            z = keuntungan(x)  # Hitung keuntungan pasangan tersebut.
            pemakaian = A @ x  # Hitung ruas kiri kedua kendala.
            layak = feasible(x)  # Tentukan apakah pasangan memenuhi semua kendala.
            print(f"{x1:3d} {x2:3d} {z:7.2f} {pemakaian[0]:12.2f} {pemakaian[1]:12.2f}    {layak}")  # Tampilkan hasil pemeriksaan.
            if layak and z > terbaik_z:  # Pilih hanya kandidat layak dengan keuntungan lebih tinggi.
                terbaik_x = x.copy()  # Simpan pasangan kandidat terbaik.
                terbaik_z = z  # Simpan nilai keuntungan terbaik.
    if terbaik_x is None:  # Pembulatan terdekat bisa gagal pada model lain.
        raise RuntimeError("Tidak ada kombinasi pembulatan terdekat yang feasible.")  # Laporkan kegagalan heuristik.
    return terbaik_x, terbaik_z  # Kembalikan solusi pembulatan dan keuntungannya.

x_round, z_round = round_off(x_lp)  # Jalankan metode Round Off.
print(f"Hasil Round Off: x1 = {x_round[0]}, x2 = {x_round[1]}, Z = {z_round:.2f}")  # Hasilnya (4, 1), Z = 34.
print("Round Off ini tidak menjamin solusi integer optimal global.")  # Kandidatnya hanya pembulatan terdekat.


ROUND OFF: KOMBINASI PEMBULATAN TERDEKAT
 x1  x2       Z    Kendala 1    Kendala 2    Feasible
  3   1   27.00         9.00        23.00    True
  3   2   33.00        12.00        28.00    True
  4   1   34.00        11.00        29.00    True
  4   2   40.00        14.00        34.00    False
Hasil Round Off: x1 = 4, x2 = 1, Z = 34.00
Round Off ini tidak menjamin solusi integer optimal global.


4. Branch and Bound atau Branch Algorithm
Setiap node menyelesaikan LP dengan kendala asli dan batas tambahan dari cabang induknya. Algoritma menyimpan cabang yang belum diperiksa dalam antrean. Cabang bawah diperiksa sebelum cabang atas.

    Jika solusi memuat variabel pecahan $x_j$, node dibagi menjadi $x_j\le\lfloor x_j\rfloor$ dan $x_j\ge\lceil x_j\rceil$. Kedua cabang ini mencakup seluruh kemungkinan bilangan bulat untuk variabel tersebut.

    Cabang dihentikan jika tidak feasible, jika batas atasnya tidak bisa mengalahkan solusi integer terbaik, atau jika solusi LP-nya sudah integer. Proses selesai ketika antrean kosong.

    Urutan dan nomor node kode berbeda dari label A–F pada slide. Model dan hasilnya tetap sama. Kode tidak memanggil solver integer otomatis: percabangan, pemeriksaan integer, dan bound ditulis dalam fungsi branch_and_bound.

In [4]:
def branch_and_bound():  # Mencari solusi integer terbaik melalui percabangan LP.
    antrean = deque([(0, [(0, None), (0, None)], "Akar")])  # Node pertama berisi model asli.
    nomor_berikut = 1  # Nomor unik untuk cabang baru.
    terbaik_x = None  # Belum ditemukan solusi integer.
    terbaik_z = -np.inf  # Keuntungan solusi integer terbaik yang sudah ditemukan.
    riwayat = []  # Simpan hasil setiap node agar langkah algoritma bisa diperiksa.
    print("\nBRANCH AND BOUND: JEJAK PERHITUNGAN")  # Judul metode kedua.
    while antrean:  # Teruskan sampai semua cabang selesai diperiksa atau dipangkas.
        nomor, batas, jalur = antrean.popleft()  # Ambil cabang terlama: urutan breadth-first.
        hasil = selesaikan_lp(batas)  # Selesaikan relaksasi LP pada cabang itu.
        if hasil.status == 2:  # Status 2 berarti kendala cabang tidak memiliki solusi feasible.
            print(f"Node {nomor}: {jalur} -> tidak feasible; hentikan cabang.")  # Laporkan cabang yang gugur.
            riwayat.append({"node": nomor, "jalur": jalur, "x": None, "z": None, "status": "Tidak feasible"})  # Catat kegagalan kelayakan.
            continue  # Beralih ke cabang berikutnya.
        if not hasil.success:  # Kegagalan lain tidak boleh dianggap sama dengan tidak feasible.
            raise RuntimeError(hasil.message)  # Laporkan jika solver gagal karena alasan lain.
        x = hasil.x.copy()  # Ambil salinan solusi LP pada cabang yang sedang diperiksa.
        x[np.abs(x) <= TOL] = 0.0  # Rapikan nilai yang sangat dekat nol agar tidak tampil sebagai -0.0000.
        batas_atas = keuntungan(x)  # Nilai LP menjadi batas atas keuntungan integer pada cabang ini.
        print(f"Node {nomor}: {jalur}")  # Tampilkan syarat tambahan pada cabang.
        print(f"  x1 = {x[0]:.4f}; x2 = {x[1]:.4f}; batas atas Z = {batas_atas:.4f}")  # Tampilkan solusi LP cabang.
        catatan = {"node": nomor, "jalur": jalur, "x": x.tolist(), "z": batas_atas}  # Siapkan catatan node.
        riwayat.append(catatan)  # Simpan catatan sebelum menentukan tindakan selanjutnya.
        if batas_atas <= terbaik_z + TOL:  # Cabang tidak bisa mengalahkan solusi integer terbaik saat ini.
            catatan["status"] = "Dipangkas oleh bound"  # Tandai pemangkasan berdasarkan batas atas.
            print("  Bound tidak lebih baik daripada solusi terbaik; hentikan cabang.")  # Jelaskan alasan pemangkasan.
            continue  # Tidak perlu membuat cabang dari node ini.
        pecahan = np.flatnonzero(np.abs(x - np.rint(x)) > TOL)  # Cari indeks variabel yang belum bulat.
        if len(pecahan) == 0:  # Semua variabel sudah mendekati bilangan bulat.
            terbaik_x = np.rint(x).astype(int)  # Ubah solusi integer dari solver ke tipe int.
            terbaik_z = keuntungan(terbaik_x)  # Perbarui keuntungan integer terbaik.
            catatan["status"] = "Solusi integer"  # Node selesai karena solusinya sudah integer.
            print(f"  Solusi integer ditemukan; keuntungan terbaik sekarang {terbaik_z:.2f}.")  # Laporkan pembaruan.
            continue  # Tidak perlu mencabangkan solusi integer.
        j = int(pecahan[0])  # Pilih variabel pecahan pertama: x1 terlebih dahulu, lalu x2.
        bawah = floor(x[j])  # Batas integer di bawah nilai pecahan.
        atas = ceil(x[j])  # Batas integer di atas nilai pecahan.
        catatan["status"] = "Dicabangkan"  # Node akan dibagi menjadi dua submasalah.
        print(f"  Cabangkan: x{j + 1} <= {bawah} atau x{j + 1} >= {atas}.")  # Kedua cabang mencakup semua kemungkinan integer.
        minimum, maksimum = batas[j]  # Ambil batas variabel yang sudah berlaku dari cabang induk.
        kiri = batas.copy()  # Buat salinan batas untuk cabang pembulatan bawah.
        kiri[j] = (minimum, bawah if maksimum is None else min(maksimum, bawah))  # Tambahkan xj <= floor(xj).
        kanan = batas.copy()  # Buat salinan batas untuk cabang pembulatan atas.
        kanan[j] = (max(minimum, atas), maksimum)  # Tambahkan xj >= ceil(xj).
        antrean.append((nomor_berikut, kiri, f"{jalur} -> x{j + 1} <= {bawah}"))  # Masukkan cabang bawah ke antrean.
        nomor_berikut += 1  # Siapkan nomor untuk cabang selanjutnya.
        antrean.append((nomor_berikut, kanan, f"{jalur} -> x{j + 1} >= {atas}"))  # Masukkan cabang atas ke antrean.
        nomor_berikut += 1  # Perbarui nomor agar setiap node berbeda.
    if terbaik_x is None:  # Periksa apakah ada solusi integer yang ditemukan.
        raise RuntimeError("Tidak ditemukan solusi integer feasible.")  # Laporkan jika model tidak punya solusi integer.
    return terbaik_x, terbaik_z, riwayat  # Kembalikan solusi terbaik dan seluruh jejak percabangan.

x_branch, z_branch, riwayat_branch = branch_and_bound()  # Jalankan algoritma percabangan sendiri.
print(f"Hasil Branch and Bound: x1 = {x_branch[0]}, x2 = {x_branch[1]}, Z = {z_branch:.2f}")  # Hasil optimal integer (5, 0), Z = 35.


BRANCH AND BOUND: JEJAK PERHITUNGAN
Node 0: Akar
  x1 = 3.7500; x2 = 1.5000; batas atas Z = 35.2500
  Cabangkan: x1 <= 3 atau x1 >= 4.
Node 1: Akar -> x1 <= 3
  x1 = 3.0000; x2 = 2.0000; batas atas Z = 33.0000
  Solusi integer ditemukan; keuntungan terbaik sekarang 33.00.
Node 2: Akar -> x1 >= 4
  x1 = 4.0000; x2 = 1.2000; batas atas Z = 35.2000
  Cabangkan: x2 <= 1 atau x2 >= 2.
Node 3: Akar -> x1 >= 4 -> x2 <= 1
  x1 = 4.1667; x2 = 1.0000; batas atas Z = 35.1667
  Cabangkan: x1 <= 4 atau x1 >= 5.
Node 4: Akar -> x1 >= 4 -> x2 >= 2 -> tidak feasible; hentikan cabang.
Node 5: Akar -> x1 >= 4 -> x2 <= 1 -> x1 <= 4
  x1 = 4.0000; x2 = 1.0000; batas atas Z = 34.0000
  Solusi integer ditemukan; keuntungan terbaik sekarang 34.00.
Node 6: Akar -> x1 >= 4 -> x2 <= 1 -> x1 >= 5
  x1 = 5.0000; x2 = 0.0000; batas atas Z = 35.0000
  Solusi integer ditemukan; keuntungan terbaik sekarang 35.00.
Hasil Branch and Bound: x1 = 5, x2 = 0, Z = 35.00


5. Perbandingan hasil
LP menghasilkan keuntungan tertinggi ketika pecahan diperbolehkan. Round Off menghasilkan solusi integer yang layak, sedangkan Branch and Bound mencari solusi integer optimal melalui percabangan.


In [5]:
print("\nPERBANDINGAN HASIL")  # Judul kesimpulan.
print(f"LP kontinu       : x1 = {x_lp[0]:.4f}; x2 = {x_lp[1]:.4f}; Z = {z_lp:.4f}")  # LP boleh menghasilkan pecahan.
print(f"Round Off        : x1 = {x_round[0]}; x2 = {x_round[1]}; Z = {z_round:.2f}")  # Solusi dari pembulatan terdekat.
print(f"Branch and Bound : x1 = {x_branch[0]}; x2 = {x_branch[1]}; Z = {z_branch:.2f}")  # Solusi integer terbaik.
print("Kesimpulan: produksi 5 drum-dryer dan 0 spray-dryer menghasilkan keuntungan maksimum 35.")  # Gunakan satuan keuntungan dari soal; jangan mengasumsikan rupiah.


PERBANDINGAN HASIL
LP kontinu       : x1 = 3.7500; x2 = 1.5000; Z = 35.2500
Round Off        : x1 = 4; x2 = 1; Z = 34.00
Branch and Bound : x1 = 5; x2 = 0; Z = 35.00
Kesimpulan: produksi 5 drum-dryer dan 0 spray-dryer menghasilkan keuntungan maksimum 35.
